In [1]:
from langgraph.graph.message import MessagesState
from langgraph.graph.state import StateGraph, START, END
from langchain_anthropic import ChatAnthropic
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [2]:
class GraphState(MessagesState):
    summary:str
    image: dict
    audio:bytes
    response_type:str
    context: str

In [3]:
from pydantic import BaseModel , Field
class Response(BaseModel):
    answer:str = Field(description="Here response of the query should be written")
    response_type:str = Field(description = "Here response type from either text or audio")


In [4]:
load_dotenv()
llm = ChatGroq(model="openai/gpt-oss-120b").with_structured_output(Response)

In [5]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage
import json

In [10]:
async def answer_question(state):
    response_obj = await llm.ainvoke(state["messages"]) 
    
    # 2. Convert Pydantic object to dict or string
    # LangGraph nodes must return a dict to update the state
    content = json.dumps(response_obj.dict())
    
    return {
        "messages": [AIMessage(content=content)],
        "response_type": response_obj.response_type
    }

In [11]:
graph = StateGraph(GraphState)
graph.add_node("answer_question", answer_question)
graph.add_edge(START, "answer_question")
graph.add_edge("answer_question", END)

graph_obj = graph.compile()

In [12]:
async def answer(query:str):
    return await graph_obj.ainvoke({
        "messages": [
    SystemMessage("You are a helpful assistant and your job is to give a reply to the user query and its your choice to give the response in either audio or text. You must give the structured output with keys of answer and response_type"),
    HumanMessage(content=query)
]
    })


In [15]:
response = await answer("Whos the current president of Pakistan")

C:\Users\Awais\AppData\Local\Temp\ipykernel_15220\667355158.py:6: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  content = json.dumps(response_obj.dict())


In [17]:
response['response_type']

'text'